# Add Registrar's Notation to all existing companies

<b> Purpose: Add Registrar's Notation filing to all existing BENs.</b>

This is a one time (python) script to be run at a given date/time.<br>
Set the configuration (client_id, client_secret, url(s)) for a scpecific environment.<br>
Get access token for authorization.


In [ ]:
%pip install cloud-sql-python-connector[pg8000] sqlalchemy ipython-sql

from dotenv import load_dotenv, find_dotenv

# this will load all the envars from a .env file located in the project root (api)
load_dotenv(find_dotenv())

%load_ext sql

In [ ]:
import os
import sqlalchemy
from google.cloud.sql.connector import Connector, IPTypes

# GCP configuration
INSTANCE_CONNECTION_NAME = os.getenv("CLOUDSQL_INSTANCE_CONNECTION_NAME")
DATABASE_USERNAME = os.getenv("DATABASE_USERNAME")
DATABASE_NAME = os.getenv("DATABASE_NAME")

# Initialize connector
connector = Connector(refresh_strategy="LAZY")

def getconn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DATABASE_USERNAME,
        db=DATABASE_NAME,
        enable_iam_auth=True,
        ip_type=IPTypes.PUBLIC
    )

# Create SQLAlchemy engine
engine = sqlalchemy.create_engine(
    "postgresql+pg8000://",
    creator=getconn
)

In [ ]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text("SELECT current_user;"))
    print(result.fetchall())

In [ ]:
import requests
import os
from datetime import datetime

# token_url, client_id, client_secret, base_url - update based on environment
token_url = os.getenv('ACCOUNT_SVC_AUTH_URL')
client_id = os.getenv('ACCOUNT_SVC_CLIENT_ID')
client_secret = os.getenv('ACCOUNT_SVC_CLIENT_SECRET')
base_url = os.getenv('LEGAL_API_BASE_URL')

header = {
    "Content-Type": "application/x-www-form-urlencoded"
}

data = 'grant_type=client_credentials'

res = requests.post(token_url, data, auth=(client_id, client_secret), headers=header)

# Check the status code of the response
if res.status_code == 200:
    print(f"Access token returned successfully : {base_url}")
    token = res.json()["access_token"]
else:
    print(f"Failed to make POST request. Status code: {res.status_code}")
    print(res.text)  # Print the error message if the request fails



Call API (POST) endpoint to create Registrar's Notation filing for businesses.

In [ ]:
from urllib.parse import urljoin
from registrars_notation_identifiers import businesses

current_date = datetime.now().date().isoformat()
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer " + token
}

successful_identifiers = []
failed_identifiers = []
skipped_identifiers = []

for identifier in businesses:
    with engine.connect() as conn:
        filing_data = {
            "filing": {
                "header": {
                    "name": "registrarsNotation",
                    "date": current_date,
                    "certifiedBy": "system"
                },
                "business": {
                    "identifier": identifier,
                    "legalType": "BEN"
                },
                "registrarsNotation": {
                    "orderDetails": (
                        "This is a test to add Registrar's Notation to a COOP"
                        "second line of the order details to test the character limit of 2000 characters. "
                        "\"This company is a test company and, as such, has purposes that include conducting its business "
                        "in a responsible and sustainable manner and promoting one or more public benefits\" to "
                        "\"This company is a test company and, as such, is committed to conducting its business in a "
                        "responsible and sustainable manner and promoting one or more public benefits\"."
                    )
                }
            }
        }

    filing_url = urljoin(base_url, f"/api/v2/businesses/{identifier}/filings")
    response = requests.post(filing_url, headers=headers, json=filing_data)

    if response.status_code == 201:
        successful_identifiers.append(identifier)
    else:
        failed_identifiers.append(identifier)
        print(
            f"Failed to make POST request. Status code: {response.status_code} "
            f"for {identifier}"
        )

print("Successfully filed Registrar Notation for:", successful_identifiers)
print("Failed to file Registrar Notation for:", failed_identifiers)
